# Patrimônio dos candidatos nas eleições de 2018 e 2022

Pipeline de dados com **DuckDB** e **dbt** sobre os dados abertos do TSE.

**Objetivo:** transformar os CSVs de candidaturas e de bens declarados em um modelo dimensional
(*star schema*) pronto para análise.

**Escopo:** eleições gerais de 2018 e 2022, apenas 1º turno e candidaturas com situação APTO.

| etapa | ferramenta | onde fica |
|---|---|---|
| Extração dos CSVs (staging) | DuckDB + Python | este notebook → schema `bronze` |
| Limpeza e tipagem | dbt | `eleicoes/models/silver` → schema `silver` |
| Modelo dimensional | dbt | `eleicoes/models/gold` → schema `gold` |

In [ ]:
# dbt init

## 1. DuckDB

Banco de dados **analítico** (OLAP) que roda **dentro do processo** (como o SQLite), sem servidor.

- Armazenamento colunar e execução vetorizada: rápido para agregações sobre milhões de linhas.
- Lê CSV, Parquet e JSON diretamente com SQL (`read_csv_auto`, `read_parquet`).
- Todo o banco fica em um único arquivo: `eleicoes.duckdb`.

In [2]:
import duckdb

duckdb.sql("SELECT 42").show()

┌───────┐
│  42   │
│ int32 │
├───────┤
│    42 │
└───────┘



## 2. Os dados brutos (TSE)

Fonte: [Portal de Dados Abertos do TSE](https://dadosabertos.tse.jus.br/), arquivos `*_BRASIL.csv`
(separador `;`, encoding `latin-1`), em `eleicoes/data/raw/`.

| arquivo | conteúdo | grão |
|---|---|---|
| `consulta_cand_<ano>` | dados da candidatura e do candidato (50 colunas) | uma candidatura por turno |
| `bem_candidato_<ano>` | bens declarados (19 colunas) | um bem de uma candidatura |

As duas tabelas se ligam pela coluna `SQ_CANDIDATO` (identificador da candidatura, muda a cada eleição).
Para identificar a **mesma pessoa** em eleições diferentes é preciso usar o CPF.

In [ ]:
# Mostrando como são os arquivos csv originais
import pandas as pd

# mostra números sem notação científic
pd.options.display.float_format = '{:.2f}'.format

df_bens = pd.read_csv('eleicoes/data/raw/bem_candidato_2018/bem_candidato_2018_BRASIL.csv', sep=';', encoding='latin-1')
df_bens.head()

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,SG_UF,SG_UE,NM_UE,SQ_CANDIDATO,NR_ORDEM_BEM_CANDIDATO,CD_TIPO_BEM_CANDIDATO,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO,DT_ULT_ATUAL_BEM_CANDIDATO,HH_ULT_ATUAL_BEM_CANDIDATO
0,13/11/2024,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,07/10/2018,PI,PI,PIAUÍ,180000614002,6,14,Terra nua,"Uma gleba de terra com área de 15 hectares, si...","123304,68",09/04/2021,09:18:22
1,13/11/2024,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,07/10/2018,PI,PI,PIAUÍ,180000614002,15,14,Terra nua,"Uma gleba de terra com área de 115 hectares, s...","19000,00",09/04/2021,09:18:22
2,13/11/2024,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,07/10/2018,PI,PI,PIAUÍ,180000614006,3,14,Terra nua,"50% de uma gleba de terras 379.97.00HA, santa ...","2500,00",09/04/2021,09:18:36
3,13/11/2024,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,07/10/2018,PI,PI,PIAUÍ,180000614006,4,14,Terra nua,"50% de uma gleba de terras, 75.90.00HA, fazend...","1500,00",09/04/2021,09:18:36
4,13/11/2024,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,07/10/2018,PI,PI,PIAUÍ,180000614008,2,14,Terra nua,"Quatro lotes de terreno, qd-152, loteam. Belo ...","1110,35",09/04/2021,09:18:26


In [8]:
df_bens.columns.to_list()

['DT_GERACAO',
 'HH_GERACAO',
 'ANO_ELEICAO',
 'CD_TIPO_ELEICAO',
 'NM_TIPO_ELEICAO',
 'CD_ELEICAO',
 'DS_ELEICAO',
 'DT_ELEICAO',
 'SG_UF',
 'SG_UE',
 'NM_UE',
 'SQ_CANDIDATO',
 'NR_ORDEM_BEM_CANDIDATO',
 'CD_TIPO_BEM_CANDIDATO',
 'DS_TIPO_BEM_CANDIDATO',
 'DS_BEM_CANDIDATO',
 'VR_BEM_CANDIDATO',
 'DT_ULT_ATUAL_BEM_CANDIDATO',
 'HH_ULT_ATUAL_BEM_CANDIDATO']

In [9]:
# Candidatos

df_candidatos = pd.read_csv('eleicoes/data/raw/consulta_cand_2018/consulta_cand_2018_BRASIL.csv', sep=';', encoding='latin-1')
df_candidatos.head()


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO
0,13/11/2024,10:55:17,2018,2,ELEIÇÃO ORDINÁRIA,1,297,Eleições Gerais Estaduais 2018,07/10/2018,ESTADUAL,...,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-1,#NULO#
1,13/11/2024,10:55:17,2018,2,ELEIÇÃO ORDINÁRIA,1,297,Eleições Gerais Estaduais 2018,07/10/2018,ESTADUAL,...,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-1,#NULO#
2,13/11/2024,10:55:17,2018,2,ELEIÇÃO ORDINÁRIA,1,297,Eleições Gerais Estaduais 2018,07/10/2018,ESTADUAL,...,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,4,NÃO ELEITO
3,13/11/2024,10:55:17,2018,2,ELEIÇÃO ORDINÁRIA,1,297,Eleições Gerais Estaduais 2018,07/10/2018,ESTADUAL,...,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-1,#NULO#
4,13/11/2024,10:55:17,2018,2,ELEIÇÃO ORDINÁRIA,1,297,Eleições Gerais Estaduais 2018,07/10/2018,ESTADUAL,...,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-4,NÃO DIVULGÁVEL,-1,#NULO#


In [10]:
df_candidatos.columns.to_list()

['DT_GERACAO',
 'HH_GERACAO',
 'ANO_ELEICAO',
 'CD_TIPO_ELEICAO',
 'NM_TIPO_ELEICAO',
 'NR_TURNO',
 'CD_ELEICAO',
 'DS_ELEICAO',
 'DT_ELEICAO',
 'TP_ABRANGENCIA',
 'SG_UF',
 'SG_UE',
 'NM_UE',
 'CD_CARGO',
 'DS_CARGO',
 'SQ_CANDIDATO',
 'NR_CANDIDATO',
 'NM_CANDIDATO',
 'NM_URNA_CANDIDATO',
 'NM_SOCIAL_CANDIDATO',
 'NR_CPF_CANDIDATO',
 'DS_EMAIL',
 'CD_SITUACAO_CANDIDATURA',
 'DS_SITUACAO_CANDIDATURA',
 'TP_AGREMIACAO',
 'NR_PARTIDO',
 'SG_PARTIDO',
 'NM_PARTIDO',
 'NR_FEDERACAO',
 'NM_FEDERACAO',
 'SG_FEDERACAO',
 'DS_COMPOSICAO_FEDERACAO',
 'SQ_COLIGACAO',
 'NM_COLIGACAO',
 'DS_COMPOSICAO_COLIGACAO',
 'SG_UF_NASCIMENTO',
 'DT_NASCIMENTO',
 'NR_TITULO_ELEITORAL_CANDIDATO',
 'CD_GENERO',
 'DS_GENERO',
 'CD_GRAU_INSTRUCAO',
 'DS_GRAU_INSTRUCAO',
 'CD_ESTADO_CIVIL',
 'DS_ESTADO_CIVIL',
 'CD_COR_RACA',
 'DS_COR_RACA',
 'CD_OCUPACAO',
 'DS_OCUPACAO',
 'CD_SIT_TOT_TURNO',
 'DS_SIT_TOT_TURNO']

## 3. Staging area (`bronze`)

Os CSVs são carregados **sem transformação** no schema `bronze` do DuckDB. Essa é a única etapa
fora do dbt: o dbt lê essas tabelas como *sources*.

Resultado: `bronze.candidatos` (58.609 linhas) e `bronze.bens` (186.086 linhas).

In [11]:
import duckdb

# Cria ou abre um banco de dados persistente
con = duckdb.connect("eleicoes.duckdb")


In [14]:
# Transformando os arquivos csv originais em tabelas do DuckDB -- Usando bronze como staging area

# Cria o schema Bronze
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

# Lê o CSV e armazena os dados em uma tabela
con.execute("""
    CREATE OR REPLACE TABLE bronze.bens AS
    SELECT *
    FROM read_csv_auto([
    'eleicoes/data/raw/bem_candidato_2018/bem_candidato_2018_BRASIL.csv',
    'eleicoes/data/raw/bem_candidato_2022/bem_candidato_2022_BRASIL.csv'
    ],delim = ';', encoding = 'latin-1')
""")

con.execute("""
    CREATE OR REPLACE TABLE bronze.candidatos AS
    SELECT *
    FROM read_csv_auto([
    'eleicoes/data/raw/consulta_cand_2018/consulta_cand_2018_BRASIL.csv',
    'eleicoes/data/raw/consulta_cand_2022/consulta_cand_2022_BRASIL.csv'
    ],delim = ';', encoding = 'latin-1')
""")


In [16]:
# Consulta a tabela criada
result = con.execute("""
    SELECT *
    FROM bronze.bens
""").df()

result.head()

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,SG_UF,SG_UE,NM_UE,SQ_CANDIDATO,NR_ORDEM_BEM_CANDIDATO,CD_TIPO_BEM_CANDIDATO,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO,DT_ULT_ATUAL_BEM_CANDIDATO,HH_ULT_ATUAL_BEM_CANDIDATO
0,2024-11-13,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,2018-10-07,PI,PI,PIAUÍ,180000614002,6,14,Terra nua,"Uma gleba de terra com área de 15 hectares, si...","123304,68",2021-04-09,09:18:22
1,2024-11-13,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,2018-10-07,PI,PI,PIAUÍ,180000614002,15,14,Terra nua,"Uma gleba de terra com área de 115 hectares, s...","19000,00",2021-04-09,09:18:22
2,2024-11-13,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,2018-10-07,PI,PI,PIAUÍ,180000614006,3,14,Terra nua,"50% de uma gleba de terras 379.97.00HA, santa ...","2500,00",2021-04-09,09:18:36
3,2024-11-13,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,2018-10-07,PI,PI,PIAUÍ,180000614006,4,14,Terra nua,"50% de uma gleba de terras, 75.90.00HA, fazend...","1500,00",2021-04-09,09:18:36
4,2024-11-13,10:55:11,2018,2,Eleição Ordinária,297,Eleições Gerais Estaduais 2018,2018-10-07,PI,PI,PIAUÍ,180000614008,2,14,Terra nua,"Quatro lotes de terreno, qd-152, loteam. Belo ...","1110,35",2021-04-09,09:18:26


In [17]:
# Consulta a tabela criada
result = con.execute("""
    SELECT DISTINCT(ANO_ELEICAO)
    FROM bronze.bens
""").df()

result.head()

,ANO_ELEICAO
0,2018
1,2022


In [21]:
con.sql("SELECT NM_CANDIDATO FROM bronze.candidatos").show()

┌─────────────────────────────────────┐
│            NM_CANDIDATO             │
│               varchar               │
├─────────────────────────────────────┤
│ GILMAR DE ASSIS                     │
│ MARIA PEREGRINA SOUZA E SILVA       │
│ PABLO YURI RAIOL SANTANA            │
│ LUCIENE BEATRIZ FACHINELLI BARBOSA  │
│ BRUNO AUGUSTO DE JESUS FERREIRA     │
│ JOELISIA MOREIRA FEITOSA FILHA      │
│ JOÃO BOSCO JARDIM DOS SANTOS        │
│ INGRID MICHELLE DA SILVA            │
│ SEBASTIÃO DE BARROS QUINTÃO         │
│ MARINA MARTA DA SILVA MENDES        │
│              ·                      │
│              ·                      │
│              ·                      │
│ SIRIO DE NAZARE AMARAL PINHEIRO     │
│ WANDERSON DA SILVA SOUZA            │
│ RUBENILSON JESUS DE SOUZA           │
│ MAURO CESAR DE OLIVEIRA             │
│ HOMERO FLAVIO DA SILVA              │
│ OZENY DE SOUZA BRITO                │
│ JACKELINE DE OLIVEIRA MACHADO       │
│ JOCYLAINE CALUMBIA PINTO DOS SANTOS │


In [41]:
con.sql("SELECT DS_TIPO_BEM_CANDIDATO FROM bronze.bens").show()

┌──────────────────────────────────────────────────────────────┐
│                    DS_TIPO_BEM_CANDIDATO                     │
│                           varchar                            │
├──────────────────────────────────────────────────────────────┤
│ Terra nua                                                    │
│ Terra nua                                                    │
│ Terra nua                                                    │
│ Terra nua                                                    │
│ Terra nua                                                    │
│ Terra nua                                                    │
│ Terra nua                                                    │
│ Terra nua                                                    │
│ Terra nua                                                    │
│ Terra nua                                                    │
│     ·                                                        │
│     ·                  

## 4. dbt (data build tool)

Ferramenta de **transformação** (o "T" do ELT). Cada modelo é um arquivo `.sql` com um `SELECT`;
o dbt cria a tabela ou view, resolve a ordem de execução e testa os dados.

- **SQL + Jinja**: `{{ ref('modelo') }}` e `{{ source('fonte', 'tabela') }}` criam as dependências.
- **DAG**: o dbt monta o grafo de dependências a partir dos `ref`/`source` e executa na ordem certa.
- **Testes**: `unique`, `not_null`, `relationships`, `accepted_values` e testes SQL próprios.
- **Documentação**: descrições em YAML viram um site (`dbt docs`).

Estrutura do projeto:

```
eleicoes/
├── dbt_project.yml          configuração do projeto
├── profiles.yml             conexão com o banco
├── models/
│   ├── sources.yml          tabelas de origem (schema bronze)
│   ├── silver/              limpeza e tipagem  + schema.yml
│   └── gold/                star schema        + schema.yml
├── macros/                  funções Jinja reutilizáveis
└── tests/                   testes SQL próprios
```

### profiles.yml

Define **onde** o dbt se conecta. Usamos o adaptador `dbt-duckdb` apontando para o arquivo do banco.

```yaml
eleicoes:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: '../eleicoes.duckdb'   # relativo à pasta eleicoes/, de onde o dbt é executado
      schema: bronze
```

O dbt procura o `profiles.yml` na pasta onde é executado e depois em `~/.dbt/`.
Por isso ele fica dentro de `eleicoes/`.

### dbt_project.yml

Define **como** os modelos são materializados, por pasta:

```yaml
models:
  eleicoes:
    silver:
      +schema: silver
      +materialized: table
    gold:
      +schema: gold
      +materialized: table
```

### sources.yml

Declara as tabelas da staging area para que os modelos usem `{{ source('eleicoes', 'bens') }}`
em vez de nomes fixos:

```yaml
sources:
  - name: eleicoes
    schema: bronze
    tables:
      - name: bens
      - name: candidatos
```

### models

**Silver**: limpeza, tipagem e filtros. Uma tabela por fonte.

| modelo | o que faz |
|---|---|
| `silver_candidatos` | mantém só o 1º turno e candidaturas APTO (52.343 linhas) |
| `silver_bens` | converte o valor `"1234,56"` para `DECIMAL(18,2)`, ajusta tipos e descarta bens sem tipo (186.086 linhas) |

**Gold**: modelo dimensional (*star schema*) para análise.

```
                 gold_dim_eleicao
                        |
gold_dim_tipo_bem -- gold_fact_bem_candidato -- gold_dim_candidatura
                        |
                 gold_dim_pessoa
```

| modelo | grão | linhas |
|---|---|---|
| `gold_fact_bem_candidato` | um bem declarado por uma candidatura | 174.358 |
| `gold_dim_candidatura` | uma candidatura (`sq_candidato`) | 52.343 |
| `gold_dim_pessoa` | uma pessoa (`sk_pessoa`, derivada do CPF) | 46.609 |
| `gold_dim_tipo_bem` | um tipo de bem do TSE + macrocategoria | 50 |
| `gold_dim_eleicao` | uma eleição (`cd_eleicao`) | 6 |

Exemplo de modelo (`gold_dim_tipo_bem.sql`): só um `SELECT`, sem `CREATE TABLE`.

```sql
SELECT
    CD_TIPO_BEM_CANDIDATO             AS cd_tipo_bem,
    ANY_VALUE(DS_TIPO_BEM_CANDIDATO)  AS ds_tipo_bem,
    CASE
        WHEN CD_TIPO_BEM_CANDIDATO BETWEEN 1 AND 19 THEN 'Imóveis'
        ...
    END                               AS ds_categoria
FROM {{ ref('silver_bens') }}
GROUP BY CD_TIPO_BEM_CANDIDATO
```

### macros

Funções Jinja reutilizáveis entre modelos.

**`sk_pessoa`**: chave substituta da pessoa, usada em `gold_dim_pessoa` e `gold_dim_candidatura`.
É o hash do CPF; quando o TSE não divulga o CPF (valor `-4`), usa o `SQ_CANDIDATO`.

```sql
{% macro sk_pessoa(cpf_column='NR_CPF_CANDIDATO', sq_column='SQ_CANDIDATO') %}
    hash(
        CASE
            WHEN regexp_full_match({{ cpf_column }}, '[0-9]{11}') THEN {{ cpf_column }}
            ELSE 'SQ:' || CAST({{ sq_column }} AS VARCHAR)
        END
    )
{% endmacro %}
```

Uso no modelo: `{{ sk_pessoa() }} AS sk_pessoa`.

**`generate_schema_name`**: sobrescreve a regra padrão do dbt para os modelos serem criados nos
schemas `silver` e `gold` (o padrão seria `bronze_silver` e `bronze_gold`).

### testes

Declarados no `schema.yml` de cada camada (35 no total):

```yaml
- name: sq_candidato
  data_tests:
    - unique
    - not_null
    - relationships:
        arguments:
          to: ref('gold_dim_candidatura')
          field: sq_candidato
```

E testes SQL próprios em `tests/`: a consulta deve retornar **zero linhas** para passar
(ex.: `assert_fact_bem_candidato_grain.sql` procura bens duplicados).

### Transformações (DAG)

O dbt deduz a ordem de execução a partir dos `ref()` e `source()`:

```
bronze.candidatos ──► silver_candidatos ──┬──► gold_dim_eleicao
                                          ├──► gold_dim_pessoa
                                          └──► gold_dim_candidatura ──┐
                                                                      ▼
bronze.bens ───────► silver_bens ─────────┬──► gold_fact_bem_candidato
                                          └──► gold_dim_tipo_bem
```

Para ver o grafo interativo: `dbt docs generate` e `dbt docs serve` (dentro de `eleicoes/`).

## 5. Executando o dbt

O DuckDB permite **um único processo de escrita** por arquivo. Antes de rodar o dbt é preciso
fechar a conexão aberta neste notebook.

`dbt build` executa, na ordem do DAG, os modelos e os testes de cada um.

In [51]:
# libera o arquivo para o dbt
con.close()

In [43]:
!cd eleicoes && ../.venv/bin/dbt build

18:55:25  Running with dbt=1.12.6
18:55:25  Registered adapter: duckdb=1.11.0
18:55:25  Unable to do partial parsing because saved manifest not found. Starting full parse.
18:55:26  Found 7 models, 35 data tests, 2 sources, 502 macros
18:55:26  
18:55:26  Concurrency: 1 threads (target='dev')
18:55:26  
18:55:26  1 of 42 START sql table model silver.silver_bens ............................... [RUN]
18:55:26  1 of 42 OK created sql table model silver.silver_bens .......................... [OK in 0.18s]
18:55:26  2 of 42 START sql table model silver.silver_candidatos ......................... [RUN]
18:55:27  2 of 42 OK created sql table model silver.silver_candidatos .................... [OK in 0.16s]
18:55:27  3 of 42 START test assert_silver_bens_grain .................................... [RUN]
18:55:27  3 of 42 PASS assert_silver_bens_grain .......................................... [PASS in 0.02s]
18:55:27  4 of 42 START test not_null_silver_bens_ANO_ELEICAO .........................

## 6. Consultando a camada gold

Reabrimos o banco em modo somente leitura e consultamos o star schema.

In [44]:
con = duckdb.connect("eleicoes.duckdb", read_only=True)

Patrimônio declarado por ano: total e mediana por bem.

In [50]:
con.sql("""
SELECT
    e.ano_eleicao,
    COUNT(*)            AS qt_bens,
    SUM(f.vr_bem)       AS vr_total,
    MEDIAN(f.vr_bem)    AS vr_mediana
FROM gold.gold_fact_bem_candidato f
JOIN gold.gold_dim_eleicao e USING (cd_eleicao)
GROUP BY e.ano_eleicao
ORDER BY e.ano_eleicao
""").df()

,ano_eleicao,qt_bens,vr_total,vr_mediana
0,2018,87202,2.178132e+10,36800.0
1,2022,87156,2.188711e+10,45000.0


Valor por macrocategoria de bem (`gold_dim_tipo_bem`).

In [46]:
con.sql("""
SELECT
    t.ds_categoria,
    COUNT(*)        AS qt_bens,
    SUM(f.vr_bem)   AS vr_total
FROM gold.gold_fact_bem_candidato f
JOIN gold.gold_dim_tipo_bem t USING (cd_tipo_bem)
GROUP BY t.ds_categoria
ORDER BY vr_total DESC
""").df()

,ds_categoria,qt_bens,vr_total
0,Imóveis,62530,2.056728e+10
1,Participações societárias,17960,1.031991e+10
2,Aplicações financeiras e fundos,24509,4.548055e+09
3,Outros bens e direitos,8824,2.683394e+09
4,"Veículos, aeronaves e embarcações",33877,2.425393e+09
5,Créditos e poupança vinculada,2848,1.502176e+09
6,Depósitos bancários,16131,6.088571e+08
7,Dinheiro em espécie,4895,5.589898e+08
8,Outros bens móveis,2784,4.543803e+08


Os 10 maiores patrimônios de 2022 (`gold_dim_candidatura`).

In [47]:
con.sql("""
SELECT
    c.nm_urna_candidato,
    c.ds_cargo,
    c.sg_partido,
    c.sg_uf,
    SUM(f.vr_bem) AS vr_patrimonio
FROM gold.gold_fact_bem_candidato f
JOIN gold.gold_dim_candidatura c USING (sq_candidato)
WHERE c.ano_eleicao = 2022
GROUP BY ALL
ORDER BY vr_patrimonio DESC
LIMIT 10
""").df()

,nm_urna_candidato,ds_cargo,sg_partido,sg_uf,vr_patrimonio
0,MARCOS ERMÍRIO DE MORAES,2º SUPLENTE,PSDB,GO,1.267951e+09
1,PAULO OCTÁVIO PO,GOVERNADOR,PSD,DF,6.188682e+08
2,PASTORE,1º SUPLENTE,MDB,DF,4.535955e+08
3,AILSON SOUTO,DEPUTADO ESTADUAL,PP,PA,4.484474e+08
4,ANTÍDIO LUNELLI,DEPUTADO ESTADUAL,MDB,SC,3.900336e+08
5,OTAVIANO PIVETTA,VICE-GOVERNADOR,REPUBLICANOS,MT,3.788696e+08
6,ARGENTA,GOVERNADOR,PSC,RS,3.729432e+08
7,JOÃO GONÇALVES,2º SUPLENTE,PSDB,RO,3.515887e+08
8,MONICA DELGADO,DEPUTADO ESTADUAL,PMN,MG,1.782150e+08
9,EUNÍCIO,DEPUTADO FEDERAL,MDB,CE,1.581845e+08


Maiores variações patrimoniais entre 2018 e 2022: a mesma pessoa liga as duas candidaturas pelo `sk_pessoa`.

In [48]:
con.sql("""
WITH patrimonio AS (
    SELECT
        c.sk_pessoa,
        c.ano_eleicao,
        c.nm_urna_candidato,
        SUM(f.vr_bem) AS vr_patrimonio
    FROM gold.gold_fact_bem_candidato f
    JOIN gold.gold_dim_candidatura c USING (sq_candidato)
    GROUP BY ALL
)
SELECT
    a.nm_urna_candidato,
    a.vr_patrimonio                     AS vr_2018,
    b.vr_patrimonio                     AS vr_2022,
    b.vr_patrimonio - a.vr_patrimonio   AS vr_variacao
FROM patrimonio a
JOIN patrimonio b
    ON a.sk_pessoa = b.sk_pessoa
   AND a.ano_eleicao = 2018
   AND b.ano_eleicao = 2022
ORDER BY vr_variacao DESC
LIMIT 10
""").df()

,nm_urna_candidato,vr_2018,vr_2022,vr_variacao
0,EUNÍCIO,89238675.75,1.581845e+08,68945783.04
1,ROMEU ZEMA,69752863.96,1.297953e+08,60042449.74
2,ZE JODAN,46119655.06,1.017939e+08,55674222.27
3,JOSÉ NELTO,7842656.00,4.849217e+07,40649514.57
4,JAIME BAGATTOLI,17464913.56,5.574222e+07,38277307.08
5,JOSE AUGUSTO,1618487.55,3.478523e+07,33166740.63
6,ZÉ TEIXEIRA,15904471.27,4.639216e+07,30487685.80
7,CHICO MACHADO,2485948.47,3.109619e+07,28610241.97
8,HERCÍLIO COELHO DINIZ,38844003.09,6.590305e+07,27059042.59
9,MISAEL VARELLA,20075223.44,4.306756e+07,22992335.26


Comparando duas candidaturas da mesma pessoa: mudança na cor/raça declarada.

In [49]:
con.sql("""
SELECT
    a.nm_candidato,
    a.ds_cor_raca AS cor_raca_2018,
    b.ds_cor_raca AS cor_raca_2022
FROM gold.gold_dim_candidatura a
JOIN gold.gold_dim_candidatura b
    ON a.sk_pessoa = b.sk_pessoa
   AND a.ano_eleicao = 2018
   AND b.ano_eleicao = 2022
WHERE a.ds_cor_raca <> b.ds_cor_raca
LIMIT 10
""").df()

,nm_candidato,cor_raca_2018,cor_raca_2022
0,NESTOR BEZERRA,PARDA,PRETA
1,FADI FAYEZ FARAJ,PARDA,BRANCA
2,BARBARA SOLANGE ROSA DOS SANTOS,PRETA,PARDA
3,MANOEL BARBOSA DO NASCIMENTO,PRETA,PARDA
4,JOSE ORLANDO DE MELO,PARDA,BRANCA
5,PERLY CIPRIANO,PRETA,PARDA
6,FRANCISCA HELENA FERNANDES FRANCA,PARDA,PRETA
7,LUZIANE DIAS SIMÃO,PARDA,BRANCA
8,TAYNA SENA SANTOS LIMA,PARDA,PRETA
9,WILLIANS ROCHA MONTEIRO,PARDA,BRANCA


## 8. Visualização com Apache Superset

O **Apache Superset** é uma plataforma de BI open source: conecta-se ao banco, permite explorar os
dados com SQL (SQL Lab) e montar gráficos e dashboards sem programar.

Neste projeto ele roda em um container Docker (`docker-compose.yml` e pasta `superset/`) e lê o
`eleicoes.duckdb` em modo somente leitura:

```bash
docker compose up -d --build
```

Acesse http://localhost:8088 (usuário `admin`, senha `admin`). O banco **eleicoes** já vem
cadastrado; as tabelas para análise estão no schema `gold`.

Sugestões de gráficos:

| pergunta | gráfico | dataset |
|---|---|---|
| Quanto foi declarado por ano? | Big Number / barras | `gold_fact_bem_candidato` |
| Em que categorias está o patrimônio? | Barras ou treemap por `ds_categoria` | fato + `gold_dim_tipo_bem` |
| Quais partidos declaram mais? | Barras por `sg_partido` | fato + `gold_dim_candidatura` |
| Patrimônio por gênero e cor/raça | Box plot (mediana) | fato + `gold_dim_candidatura` |

Para cruzar a fato com as dimensões, crie um dataset a partir do SQL Lab
(consulta no `README.md`).